In [6]:
# ==============================================================================
# CELDA 1: LIBRERÍAS + INGESTA PAT 2016-2017
# ==============================================================================
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PAT/'

archivos = {2016: 'PAT2016.xlsx', 2017: 'PAT2017.xlsx'}

pat_brutas = {}
for anio, nombre in archivos.items():
    pat_brutas[anio] = pd.read_excel(RUTA + nombre)
    print(f"OK | {anio} | {pat_brutas[anio].shape[0]} filas x {pat_brutas[anio].shape[1]} columnas")

print("\nColumnas 2016:", pat_brutas[2016].columns.tolist())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
OK | 2016 | 4661 filas x 83 columnas
OK | 2017 | 6273 filas x 83 columnas

Columnas 2016: ['AGNO_TERMINO', 'PA_FEC_TERMINO', 'AGNO_INGRESO', 'PA_FEC_ING', 'PA_ID', 'REGION_ACTA', 'ACTIVIDAD', 'PROGRAMA', 'TIPO_INSTI_FISCALIZADA', 'EE_RBD', 'EE_RSIE', 'EE_NOMBRE', 'EE_COD_REG', 'EE_COD_PRO', 'EE_COD_COM', 'EE_NOM_COM', 'EE_DEPE', 'EE_DEPE_AGRUP', 'SOST_MRUN', 'SOST_RUN', 'SOST_NOMBRE', 'N_REX_TERMINO', 'PA_TIENE_RECLAMACION', 'N_REX_RECLAMACION', 'PA_ESTADO', 'PA_INSTANCIA', 'SANCION_AMONESTACION_PRIMERA', 'SANCION_MULTA_PRIMERA', 'SANCION_INHABILIDAD_PRIMERA', 'SANCION_PRIVACION_PRIMERA', 'SANCION_REINTEGRO_PRIMERA', 'SANCION_REVOCACION_PRIMERA', 'SANCION_SOBRESEIDO_PRIMERA', 'SANCION_SUSP_SUBV_PRIMERA', 'SIN_SANCION_PRIMERA', 'PA_SANCION_INSTANCIA_1', 'MONTO_MULTA_PRIMERA', 'MONTO_REINTEGRO_PRIMERA', 'PORCENTAJE_PRIVACION_PRIMERA', 'MESES_PRIVACION_PRIMERA',

In [7]:
# ==============================================================================
# CELDA 2: AGREGACIÓN POR RBD
# ==============================================================================
pat_todas = pd.concat(pat_brutas.values(), ignore_index=True)

pat_todas['EE_RBD'] = pd.to_numeric(pat_todas['EE_RBD'], errors='coerce')
pat_todas = pat_todas.dropna(subset=['EE_RBD'])
pat_todas['EE_RBD'] = pat_todas['EE_RBD'].astype('Int64').astype(str)

agg = pat_todas.groupby('EE_RBD').agg(
    procesos_total=('PA_ID', 'count'),
    procesos_con_sancion=('SIN_SANCION_PRIMERA', lambda x: (x == 0).sum()),
    procesos_multa=('SANCION_MULTA_PRIMERA', lambda x: (x == 1).sum()),
    procesos_privacion_subvencion=('SANCION_PRIVACION_PRIMERA', lambda x: (x == 1).sum()),
).reset_index().rename(columns={'EE_RBD': 'rbd'})

print(f"Colegios con al menos 1 proceso: {len(agg)}")
print(agg.describe())

RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
agg.to_parquet(RUTA_SALIDA + 'procesos_admin_2016_17_por_rbd.parquet', index=False)
print("Guardado OK")

Colegios con al menos 1 proceso: 6031
       procesos_total  procesos_con_sancion  procesos_multa  \
count     6031.000000            6031.00000     6031.000000   
mean         1.812137               1.69093        0.764052   
std          1.177325               1.17491        0.938865   
min          1.000000               0.00000        0.000000   
25%          1.000000               1.00000        0.000000   
50%          1.000000               1.00000        1.000000   
75%          2.000000               2.00000        1.000000   
max         14.000000              13.00000       11.000000   

       procesos_privacion_subvencion  
count                    6031.000000  
mean                        0.241751  
std                         0.481761  
min                         0.000000  
25%                         0.000000  
50%                         0.000000  
75%                         0.000000  
max                         4.000000  
Guardado OK


In [8]:
# ==============================================================================
# CELDA 3 (CORREGIDA): INTEGRAR PROCESOS ADMINISTRATIVOS A LA TABLA DE ENTRENAMIENTO
# ==============================================================================
RUTA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
df_modelo = pd.read_parquet(RUTA + 'tabla_modelo_final_v3.parquet')  # v3: SIMCE+SNED+IDPS+Denuncias
pat = pd.read_parquet(RUTA + 'procesos_admin_2016_17_por_rbd.parquet')

df_modelo_v5 = pd.merge(df_modelo, pat, on='rbd', how='left', validate='one_to_one')

cols_pat = [c for c in pat.columns if c != 'rbd']
df_modelo_v5[cols_pat] = df_modelo_v5[cols_pat].fillna(0)

print(f"Filas: {len(df_modelo_v5)} (antes: {len(df_modelo)})")
print(f"Con al menos 1 proceso: {(df_modelo_v5['procesos_total']>0).sum()}")

df_modelo_v5.to_parquet(RUTA + 'tabla_modelo_final_v5.parquet', index=False)
print(df_modelo_v5.shape)

Filas: 7754 (antes: 7754)
Con al menos 1 proceso: 4016
(7754, 48)


In [10]:
# ==============================================================================
# CELDA 4: PROCESOS ADMINISTRATIVOS 2018-2022 (para validación bienio 2023-24)
# ==============================================================================
RUTA_PAT = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PAT/'

archivos_v2 = {
    2018: 'PAT2018.xlsx',
    2019: 'PAT2019.xlsx',
    2020: 'PAT2020.xlsx',
    2021: 'PAT2021.xlsx',
    2022: 'PAT2022.xlsx',
}

pat_brutas_v2 = {}
for anio, nombre in archivos_v2.items():
    pat_brutas_v2[anio] = pd.read_excel(RUTA_PAT + nombre)
    print(f"OK | {anio} | {pat_brutas_v2[anio].shape[0]} filas x {pat_brutas_v2[anio].shape[1]} columnas")

pat_todas_v2 = pd.concat(pat_brutas_v2.values(), ignore_index=True)
pat_todas_v2['EE_RBD'] = pd.to_numeric(pat_todas_v2['EE_RBD'], errors='coerce')
pat_todas_v2 = pat_todas_v2.dropna(subset=['EE_RBD'])
pat_todas_v2['EE_RBD'] = pat_todas_v2['EE_RBD'].astype('Int64').astype(str)

agg_v2 = pat_todas_v2.groupby('EE_RBD').agg(
    procesos_total=('PA_ID', 'count'),
    procesos_con_sancion=('SIN_SANCION_PRIMERA', lambda x: (x == 0).sum()),
    procesos_multa=('SANCION_MULTA_PRIMERA', lambda x: (x == 1).sum()),
    procesos_privacion_subvencion=('SANCION_PRIVACION_PRIMERA', lambda x: (x == 1).sum()),
).reset_index().rename(columns={'EE_RBD': 'rbd'})

print(f"Colegios con proceso (2018-22): {len(agg_v2)}")
RUTA_SALIDA = '/content/drive/MyDrive/INFORMATICA/1123.Tesis/Data-Scientist/PROCESADOS/'
agg_v2.to_parquet(RUTA_SALIDA + 'procesos_admin_2018_22_por_rbd.parquet', index=False)
print("Guardado OK")

OK | 2018 | 6756 filas x 83 columnas
OK | 2019 | 7045 filas x 83 columnas
OK | 2020 | 4464 filas x 83 columnas
OK | 2021 | 4469 filas x 83 columnas
OK | 2022 | 3900 filas x 83 columnas
Colegios con proceso (2018-22): 7258
Guardado OK
